In [1]:
from fasthtml.common import *
from fasthtml.js import HighlightJS
from html2text import HTML2Text
from textwrap import dedent
from json import dumps,loads
from trafilatura import html2txt, extract
from lxml.html.clean import Cleaner
import httpx, lxml
import re

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
def get_body(url):
    body = lxml.html.fromstring(httpx.get(url).text).xpath('//body')[0]
    body = Cleaner(javascript=True, style=True).clean_html(body)
    return ''.join(lxml.html.tostring(c, encoding='unicode') for c in body)

@rt('/load')
def post(sess, url:str):
    if not url: return add_toast(sess, "Please enter a valid URL", "warning")
    return set_cm(get_body(url))

def get_md(cts, extractor):
    if extractor=='traf':
        if '<article>' not in cts.lower(): cts = f'<article>{cts}</article>'
        res = extract(f'<html><body>{cts}</body></html>', output_format='markdown',
            favor_recall=True, include_tables=True, include_links=False, include_images=False, include_comments=True)
    else:
        h2t = HTML2Text(bodywidth=5000)
        h2t.ignore_links = True
        h2t.mark_code = True
        h2t.ignore_images = True
        res = h2t.handle(cts)
    def _f(m): return f'```\n{dedent(m.group(1))}\n```'
    return re.sub(r'\[code]\s*\n(.*?)\n\[/code]', _f, res or '', flags=re.DOTALL).strip()

In [3]:
docs = loader.load()

In [15]:
print(docs[0].page_content.replace("\n", " "))

         ESPN - Serving Sports Fans. Anytime. Anywhere.                                                                                                     Skip to main content               Skip to navigation                                       <  >          MenuESPN      scores    NFLNBAMLBNCAAFNHLSoccer…WNBABoxingCFLNCAACricketF1GolfHorseLLWSMMANASCARNBA G LeagueNBA Summer LeagueNCAAMNCAAWNWSLOlympicsPLLProfessional WrestlingRacingRN BBRN FBRugbySports BettingTennisX GamesUFLFantasyWatchESPN BETESPN+                    Subscribe Now      PGA TOUR LIVE: FedEx Cup Playoffs        Bundesliga        LALIGA        PFL Playoffs: Featherweights & Welterweights        MLB        In The Arena: Serena Williams        Fantasy Football Cheat Sheet   Quick Links     Little League World Series        MLB Standings        2024 US Open        2024 NFL Schedule        WNBA Rookie Tracker        Sign up: Fantasy Football        ESPN Radio: Listen Live        Watch Tennis on ESPN        Watch Golf o